In [23]:
import os

# Data was moved from Google Drive to local filesystem.
# Path is relative to the notebook's working directory (repo root).
BASE_DATA_DIR = os.path.join(".", "data")

In [24]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [25]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [26]:
respiratory_path = os.path.join(BASE_DATA_DIR, "datasets", "Respiratory_Sound_Database", "Respiratory_Sound_Database", "audio_and_txt_files")

lung_path = os.path.join(BASE_DATA_DIR, "datasets", "Audio Files")

cremad_path = os.path.join(BASE_DATA_DIR, "datasets", "AudioWAV")

In [27]:
print(os.listdir(respiratory_path)[:5])

print(os.listdir(lung_path)[:5])

print(os.listdir(cremad_path)[:5])

['178_1b3_Ar_mc_AKGC417L(1).txt', '204_7p5_Pr_mc_AKGC417L(1).txt', '176_2b3_Pl_mc_AKGC417L(1).wav', '148_1b1_Al_sc_Meditron.txt', '178_1b3_Pl_mc_AKGC417L.wav']
['BP33_Asthma,E W,P R M,43,F.wav', 'DP63_COPD,E W,P R L ,58,F.wav', 'EP30_N,N,P R M,18,F.wav', 'DP85_N,N,A R U,33,M.wav', 'DP71_N,N,P R U,36,M.wav']
['1087_IOM_ANG_XX.wav', '1079_WSI_NEU_XX.wav', '1082_IWW_ANG_XX.wav', '1084_IEO_SAD_LO.wav', '1091_IEO_SAD_LO.wav']


In [28]:
def extract_features(file_path):

    try:
        audio, sr = librosa.load(file_path, sr=None)

        # Skip empty audio
        if len(audio) == 0:
            return None

        # Avoid divide by zero
        max_value = np.max(np.abs(audio))

        if max_value == 0:
            return None

        # Normalize
        audio = audio / max_value

        # Silence Removal
        trimmed_audio, _ = librosa.effects.trim(audio)

        # Skip if audio becomes empty
        if len(trimmed_audio) < 2048:
            return None

        # MFCC Extraction
        mfcc = librosa.feature.mfcc(
            y=trimmed_audio,
            sr=sr,
            n_mfcc=40
        )

        mfcc_mean = mfcc.mean(axis=1)

        return mfcc_mean

    except Exception as e:
        print("Error:", file_path)

        return None

In [29]:
# Process Respiratory Dataset
features = []
labels = []

for file in os.listdir(respiratory_path):

    if file.endswith(".wav"):

        file_path = os.path.join(respiratory_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("respiratory")

In [30]:
#Process Lung Dataset
for file in os.listdir(lung_path):

    if file.endswith(".wav"):

        file_path = os.path.join(lung_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("lung")

In [31]:
# Process CREMAD Dataset
for file in os.listdir(cremad_path):

    if file.endswith(".wav"):

        file_path = os.path.join(cremad_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("emotion")

In [32]:
#Create Final Dataset
df = pd.DataFrame(
    features,
    columns=[f"MFCC_{i}" for i in range(40)]
)

df["label"] = labels

df


,MFCC_0,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,...,MFCC_31,MFCC_32,MFCC_33,MFCC_34,MFCC_35,MFCC_36,MFCC_37,MFCC_38,MFCC_39,label
0,-294.134430,106.292686,46.354568,45.524151,28.703033,26.142578,18.358025,15.856830,11.808986,10.392256,...,1.081882,0.327380,1.267907,0.365991,1.534480,0.691573,1.976279,1.370122,1.729002,respiratory
1,-374.055298,87.788109,55.246502,32.077686,22.256821,18.525936,16.374872,14.853044,13.540815,11.982816,...,1.513203,1.320808,1.281122,1.360069,1.337455,1.164218,1.006054,0.879924,0.657225,respiratory
2,-291.271301,153.194901,60.100471,41.074360,15.421990,8.430692,3.358740,6.331198,5.726296,6.984044,...,4.535133,5.112586,5.228918,3.420913,2.378690,0.711587,0.693194,0.362787,1.000239,respiratory
3,-306.890015,142.513535,61.176037,16.314157,15.918200,19.435120,19.368315,16.325333,7.495818,6.866500,...,3.998215,1.028025,1.926149,-3.197525,-0.387756,-2.726752,3.190512,0.444236,1.015911,respiratory
4,-377.696411,96.059845,62.758713,35.909672,23.440586,20.212894,20.737740,21.472393,20.045053,16.466190,...,0.986161,0.644100,1.031181,1.633041,2.015566,2.085112,1.921423,1.677169,1.592891,respiratory
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,-223.804550,78.360970,32.454472,19.208628,18.619974,6.784402,-0.417647,-3.189373,-1.705046,-3.356714,...,-1.602530,-1.191988,-1.806187,-2.984912,-2.927771,-3.123986,-1.895329,-4.303643,-4.821119,emotion
1984,-254.889603,50.012447,7.879866,22.467918,0.385825,-0.034878,-1.070339,-0.083858,-10.100955,-2.984109,...,-1.672996,-4.545973,-1.804348,-1.058203,-1.725346,0.644407,-2.137527,-1.554504,1.159330,emotion
1985,-145.655518,99.512665,44.774105,28.721703,13.350753,2.531798,5.441426,3.968953,-2.164299,-0.685457,...,4.766255,6.644974,5.814063,3.094983,-0.347079,-1.055710,-3.202568,-1.704240,-0.893104,emotion
1986,-142.146927,101.547737,38.021687,30.223679,11.271504,2.175860,-0.741817,0.887849,3.203635,-0.648358,...,0.165309,2.040886,4.074171,4.092439,-1.878049,-3.704716,-3.057401,-0.984545,-3.314500,emotion


In [33]:
df.shape

(1988, 41)

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1988 entries, 0 to 1987
Data columns (total 41 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MFCC_0   1988 non-null   float32
 1   MFCC_1   1988 non-null   float32
 2   MFCC_2   1988 non-null   float32
 3   MFCC_3   1988 non-null   float32
 4   MFCC_4   1988 non-null   float32
 5   MFCC_5   1988 non-null   float32
 6   MFCC_6   1988 non-null   float32
 7   MFCC_7   1988 non-null   float32
 8   MFCC_8   1988 non-null   float32
 9   MFCC_9   1988 non-null   float32
 10  MFCC_10  1988 non-null   float32
 11  MFCC_11  1988 non-null   float32
 12  MFCC_12  1988 non-null   float32
 13  MFCC_13  1988 non-null   float32
 14  MFCC_14  1988 non-null   float32
 15  MFCC_15  1988 non-null   float32
 16  MFCC_16  1988 non-null   float32
 17  MFCC_17  1988 non-null   float32
 18  MFCC_18  1988 non-null   float32
 19  MFCC_19  1988 non-null   float32
 20  MFCC_20  1988 non-null   float32
 21  MFCC_21  1988 non-null   

In [35]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df["label"])

X = df.drop("label", axis=1)

In [36]:
y

array([2, 2, 2, ..., 0, 0, 0], shape=(1988,))

In [37]:
X

,MFCC_0,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,...,MFCC_30,MFCC_31,MFCC_32,MFCC_33,MFCC_34,MFCC_35,MFCC_36,MFCC_37,MFCC_38,MFCC_39
0,-294.134430,106.292686,46.354568,45.524151,28.703033,26.142578,18.358025,15.856830,11.808986,10.392256,...,0.949883,1.081882,0.327380,1.267907,0.365991,1.534480,0.691573,1.976279,1.370122,1.729002
1,-374.055298,87.788109,55.246502,32.077686,22.256821,18.525936,16.374872,14.853044,13.540815,11.982816,...,1.781524,1.513203,1.320808,1.281122,1.360069,1.337455,1.164218,1.006054,0.879924,0.657225
2,-291.271301,153.194901,60.100471,41.074360,15.421990,8.430692,3.358740,6.331198,5.726296,6.984044,...,1.856616,4.535133,5.112586,5.228918,3.420913,2.378690,0.711587,0.693194,0.362787,1.000239
3,-306.890015,142.513535,61.176037,16.314157,15.918200,19.435120,19.368315,16.325333,7.495818,6.866500,...,1.698035,3.998215,1.028025,1.926149,-3.197525,-0.387756,-2.726752,3.190512,0.444236,1.015911
4,-377.696411,96.059845,62.758713,35.909672,23.440586,20.212894,20.737740,21.472393,20.045053,16.466190,...,2.205317,0.986161,0.644100,1.031181,1.633041,2.015566,2.085112,1.921423,1.677169,1.592891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,-223.804550,78.360970,32.454472,19.208628,18.619974,6.784402,-0.417647,-3.189373,-1.705046,-3.356714,...,-1.169181,-1.602530,-1.191988,-1.806187,-2.984912,-2.927771,-3.123986,-1.895329,-4.303643,-4.821119
1984,-254.889603,50.012447,7.879866,22.467918,0.385825,-0.034878,-1.070339,-0.083858,-10.100955,-2.984109,...,-1.346521,-1.672996,-4.545973,-1.804348,-1.058203,-1.725346,0.644407,-2.137527,-1.554504,1.159330
1985,-145.655518,99.512665,44.774105,28.721703,13.350753,2.531798,5.441426,3.968953,-2.164299,-0.685457,...,4.216758,4.766255,6.644974,5.814063,3.094983,-0.347079,-1.055710,-3.202568,-1.704240,-0.893104
1986,-142.146927,101.547737,38.021687,30.223679,11.271504,2.175860,-0.741817,0.887849,3.203635,-0.648358,...,0.926661,0.165309,2.040886,4.074171,4.092439,-1.878049,-3.704716,-3.057401,-0.984545,-3.314500


In [38]:
print(encoder.classes_)

['emotion' 'lung' 'respiratory']


In [39]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [40]:
# Build Deep Learning Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(128, activation='relu', input_shape=(40,)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

/Users/abhishekkumar/Documents/intern_aicahealth/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7036 - loss: 2.3587 - val_accuracy: 0.9025 - val_loss: 0.2708
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 697us/step - accuracy: 0.8915 - loss: 0.3435 - val_accuracy: 0.7987 - val_loss: 0.7192
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 628us/step - accuracy: 0.9159 - loss: 0.3164 - val_accuracy: 0.9497 - val_loss: 0.1374
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 612us/step - accuracy: 0.9387 - loss: 0.1750 - val_accuracy: 0.9528 - val_loss: 0.1002
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - accuracy: 0.9481 - loss: 0.1367 - val_accuracy: 0.9214 - val_loss: 0.2084
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 596us/step - accuracy: 0.9497 - loss: 0.1433 - val_accuracy: 0.9717 - val_loss: 0.0594
Epoch 7/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 581us/step - accuracy: 0.9458 - loss: 0.1549 - val_accuracy: 0.9748 - val_loss: 0.0676
Epoch 8/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step - accuracy: 0.9568 - loss: 0.1130 - val_accuracy: 0.94

In [42]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step - accuracy: 0.9849 - loss: 0.0637  
Accuracy: 0.9849246144294739
